# ISLES2024 — Final 149-patient paired four-metric evaluation (Dataset504 vs Dataset505)

**Runtime → Change runtime type → CPU (no GPU needed) → Run all.**

This notebook computes the remaining **lesion-wise F1** and **absolute lesion-count difference** for **NCCT + Tmax + CBF + CBV + MTT** with the repository's original **Panoptica 0.2-threshold** implementation. It also recomputes Dice and absolute volume difference from NIfTI masks, cross-checks per-patient nnU-Net Dice, and performs the **four-metric paired comparison against the frozen 149-patient NCCT-only baseline**. It performs **no retraining**.

It does not use unavailable organizer-specific NCCT brain masks. Therefore all outputs are **local patient-wise out-of-fold** metrics, **not official ISLES'24 hidden-test scores**.

### Scientific integrity

- Patient-wise split identity and SHA256 verified before evaluation; all 149 validation patients occur exactly once.
- Fail-closed prediction/reference geometry checks, checkpoint presence, and exact fold membership.
- Persistent per-patient case checkpoints and progress logs allow **resume after Colab disconnection**.
- Four predetermined endpoints, 20,000 paired patient bootstrap replicates, 100,000 sign-flip permutation replicates, and **Holm multiplicity correction** across the four tests.
- A frozen repository commit is used for evaluator consistency. Detailed outputs stay in your private Google Drive.

Original evaluator: https://github.com/ezequieldlrosa/isles24/blob/6b4dea6f07bf1717cca638ea1fe9345a47c3cd41/utils/eval_utils.py


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Prepare output folders and pinned source (CPU only)

A **separate** `perfusion_oof_evaluation` output directory is used, protecting the frozen Dataset504 baseline. Progress is logged to Google Drive.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, os, shutil, subprocess, sys, zipfile

ROOT = Path('/content/drive/MyDrive/HybridStrokeSeg/ISLES2024')
PREPROCESSED = ROOT / 'nnUNet_preprocessed'
RESULTS = ROOT / 'nnUNet_results'
PERF_EVAL = ROOT / 'results' / 'perfusion_oof_evaluation'
PAIRED = ROOT / 'results' / 'ncct_vs_perfusion_paired'
BASELINE = ROOT / 'results' / 'ncct_oof_evaluation' / 'isles2024_ncct_oof_case_metrics.csv'
LOG = PERF_EVAL / 'perfusion_oof_resumable.log'
REPO_COMMIT = '40e29c2e5037cfef42c2a499010a8c2a81f0e0a3'
ZIP = Path('/content/stroke_eval_pinned.zip')
REPO = Path('/content/HybridStrokeSeg-Reproduction-' + REPO_COMMIT)
PERF_EVAL.mkdir(parents=True, exist_ok=True)
PAIRED.mkdir(parents=True, exist_ok=True)

ENV = os.environ.copy()
ENV['PYTHONPATH'] = str(REPO / 'src') + os.pathsep + ENV.get('PYTHONPATH', '')
ENV['PYTHONUNBUFFERED'] = '1'

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as fp:
        for block in iter(lambda: fp.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

def run_logged(label, command):
    command = [str(v) for v in command]
    print('\n===', label, '===\n$', ' '.join(command), flush=True)
    with LOG.open('a', encoding='utf8') as lf:
        lf.write(f"\n[{datetime.now(timezone.utc).isoformat()}] {label}: $ {' '.join(command)}\n")
        lf.flush()
        with subprocess.Popen(command, cwd=REPO if REPO.exists() else None, env=ENV, stdout=subprocess.PIPE,
                              stderr=subprocess.STDOUT, bufsize=1, text=True) as proc:
            for line in proc.stdout:
                print(line, end='', flush=True)
                lf.write(line)
                lf.flush()
            return_code = proc.wait()
        lf.write(f'[{label}] exit_code={return_code}\n')
        lf.flush()
    if return_code != 0:
        raise RuntimeError(f'{label} failed (exit code {return_code}); inspect {LOG}')

for directory in (PREPROCESSED, RESULTS):
    if not directory.is_dir(): raise FileNotFoundError(directory)
if not BASELINE.is_file(): raise FileNotFoundError(f'Frozen NCCT baseline missing: {BASELINE}')

if not (REPO / 'scripts/evaluate_isles2024_nnunet_oof.py').is_file():
    run_logged('Download pinned evaluator', ['curl', '-fLsS', '--retry', '5', '--retry-delay', '4',
        f'https://github.com/NTinkicht/HybridStrokeSeg-Reproduction/archive/{REPO_COMMIT}.zip', '-o', ZIP])
    with zipfile.ZipFile(ZIP) as z: z.extractall('/content')
if not (REPO / 'scripts/evaluate_isles2024_nnunet_oof.py').is_file():
    raise RuntimeError('Expected evaluator not found in pinned repository archive')
print('Pinned evaluator commit:', REPO_COMMIT)
print('Persistent progress:', PERF_EVAL / 'ncct_oof_progress.json')
print('Persistent log:', LOG)


## 2. Verify frozen folds and 149 prediction masks

This check fails immediately if the dataset split, training checkpoints, cohort coverage, or prediction inventory is incomplete. No evaluation begins on inconsistent data.


In [ ]:
from collections import Counter
sys.path.insert(0, str(REPO / 'src'))
from hybridstrokeseg.isles2024_eval import load_splits, validate_oof_splits

NCCT_PREP = PREPROCESSED / 'Dataset504_ISLES2024_NCCT'
PERF_PREP = PREPROCESSED / 'Dataset505_ISLES2024_NCCT_PERFUSION'
PERF_TRAIN = (RESULTS / 'Dataset505_ISLES2024_NCCT_PERFUSION' /
              'nnUNetTrainer__nnUNetPlans__3d_fullres')
split_ncct = NCCT_PREP / 'splits_final.json'
split_perf = PERF_PREP / 'splits_final.json'
expected_sha = '664ed51e15374f8f9568d586fba2cb87d5157dc90986e5eca82c9c8440bf6027'
assert sha256(split_ncct) == expected_sha, 'Frozen Dataset504 split hash mismatch'
assert sha256(split_perf) == expected_sha, 'Dataset505 split differs from frozen baseline'
splits = load_splits(split_perf)
assert len(splits) == 5
cases = validate_oof_splits(splits)
assert len(cases) == 149
assert [len(s['val']) for s in splits] == [30,30,30,30,29]
for fold, sp in enumerate(splits):
    fold_dir = PERF_TRAIN / f'fold_{fold}'
    ckpt = fold_dir / 'checkpoint_final.pth'
    masks = sorted(p.name.removesuffix('.nii.gz') for p in (fold_dir / 'validation').glob('*.nii.gz'))
    assert ckpt.is_file() and ckpt.stat().st_size > 0, f'Fold {fold} has no trained final checkpoint'
    assert masks == sorted(sp['val']), f'Fold {fold} validation prediction/membership mismatch'
    assert (fold_dir / 'validation' / 'summary.json').is_file(), f'Fold {fold} lacks validation summary'
    print(f'Fold {fold}: {len(masks)} masks, checkpoint {ckpt.stat().st_size:,} bytes — PASS')
print('149-patient complete OOF coverage / exact split SHA256 — PASS')


## 3. Install the narrow evaluation dependencies and run a synthetic metric smoke test

This reuses the **wheel-only dependency strategy** of the successful NCCT-only CPU evaluator notebook. Do not install the complete nnU-Net training environment.


In [ ]:
run_logged('Install wheel-only medical imaging metric dependencies', [
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check', '--only-binary=:all:',
    'nibabel>=5.2,<6', 'connected-components-3d>=3.12,<4', 'surface-distance>=0.1,<1',
    'ruamel.yaml>=0.18,<1', 'scikit-image>=0.23,<1', 'SimpleITK>=2.3,<3',
    'rich>=13,<15', 'pandas>=2.2,<3', 'plotly>=5,<7'])
run_logged('Install pinned Panoptica 1.4.1 wheel', [
    sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
    '--only-binary=:all:', '--no-deps', 'panoptica==1.4.1'])
import importlib.util
if importlib.util.find_spec('torch') is None:
    run_logged('Install PyTorch CPU wheel', [sys.executable, '-m', 'pip', 'install',
        '--only-binary=:all:', 'torch', '--index-url', 'https://download.pytorch.org/whl/cpu'])
smoke = r"""
import numpy as np
from importlib.metadata import version
from hybridstrokeseg.isles2024_eval import evaluate_case
mask=np.zeros((7,7,7),dtype=np.uint8); mask[2:5,2:5,2:5]=1
x=evaluate_case(case_id='synthetic',fold=0,reference=mask,prediction=mask.copy(),voxel_volume_ml=0.001)
assert abs(x.dice-1)<1e-8 and abs(x.lesion_f1-1)<1e-8, x
print('Panoptica API synthetic smoke PASS:',version('panoptica'))
"""
run_logged('Pinned Panoptica smoke test', [sys.executable, '-c', smoke])


## 4. Compute **all four** Dataset505 OOF metrics, resumably

Evaluation reads the stored 149 prediction and reference NIfTI files; it neither modifies the models nor generates new predictions. Each validated case is cached with its input signature; rerunning after a disconnect resumes safely.

Expect separate local Dice, absolute volume difference (mL), lesion-wise F1, and absolute lesion-count difference values.


In [ ]:
run_logged('Dataset505 149-case four-metric local OOF evaluation', [
    sys.executable, '-u', 'scripts/evaluate_isles2024_nnunet_oof.py',
    '--dataset-id', '505', '--configuration', '3d_fullres',
    '--preprocessed-root', PREPROCESSED,
    '--results-root', RESULTS,
    '--output-dir', PERF_EVAL,
    '--bootstrap-repetitions', '10000', '--bootstrap-seed', '2026'])

# The legacy evaluator's filename has 'ncct' hard-coded. Copy, never overwrite,
# into unmistakably Dataset505-named outputs after all cases have completed.
for suffix in ('case_metrics.csv','summary.json','manifest.json'):
    old = PERF_EVAL / f'isles2024_ncct_oof_{suffix}'
    new = PERF_EVAL / f'isles2024_perfusion_oof_{suffix}'
    if not old.is_file(): raise FileNotFoundError(old)
    shutil.copy2(old, new)
print('Dataset505 case metrics:', PERF_EVAL / 'isles2024_perfusion_oof_case_metrics.csv')


## 5. Four-metric paired inference with Holm correction

The included analysis script enforces exact 149-patient pairing, fold identity, voxel-volume and reference-volume equality, and unmasked-label comparability. It saves a detailed per-patient CSV, JSON results, and a Markdown report to `results/ncct_vs_perfusion_paired`.


In [ ]:
from pathlib import Path
import hashlib
PAIR_SCRIPT = Path('/content/isles2024_paired_analysis.py')
# Pin the paired-statistics code to a specific Git commit; never use mutable main.
PAIRED_SCRIPT_COMMIT = 'a01b9ae2de966f0e0928deb6071733ac74d6f596'
run_logged('Fetch versioned paired-analysis script', [
    'curl', '-fLsS', '--retry', '5', '--retry-delay', '4',
    f'https://raw.githubusercontent.com/NTinkicht/HybridStrokeSeg-Reproduction/{PAIRED_SCRIPT_COMMIT}/scripts/compare_isles2024_ncct_perfusion_paired.py',
    '-o', PAIR_SCRIPT])
source_text = PAIR_SCRIPT.read_text(encoding='utf-8')
assert 'def holm_adjust(' in source_text, 'Wrong paired analysis source'
assert 'holm_adjusted_p_4_metrics' in source_text
expected_hash = hashlib.sha256(PAIR_SCRIPT.read_bytes()).hexdigest()
print('Paired analysis commit:', PAIRED_SCRIPT_COMMIT)
print('Paired analysis SHA256:', expected_hash)
run_logged('149-patient complete paired four-metric analysis', [
    sys.executable, '-u', PAIR_SCRIPT,
    '--baseline-csv', BASELINE,
    '--perfusion-csv', PERF_EVAL / 'isles2024_perfusion_oof_case_metrics.csv',
    '--output-dir', PAIRED,
    '--bootstrap-repetitions', '20000', '--permutation-repetitions', '100000',
    '--seed', '20261119'])
provenance = {
    'evaluator_github_commit': REPO_COMMIT,
    'evaluator_source_sha256': sha256(REPO/'src/hybridstrokeseg/isles2024_eval.py'),
    'evaluator_script_sha256': sha256(REPO/'scripts/evaluate_isles2024_nnunet_oof.py'),
    'paired_script_sha256': expected_hash,
    'frozen_split_sha256': expected_sha,
    'baseline_file_sha256': sha256(BASELINE),
    'perfusion_case_metrics_sha256': sha256(PERF_EVAL/'isles2024_perfusion_oof_case_metrics.csv'),
    'brain_mask_applied': False,
    'n_oof_patients': 149,
    'generated_at_utc': datetime.now(timezone.utc).isoformat(),
}
(PAIRED/'isles2024_paired_provenance.json').write_text(json.dumps(provenance, indent=2)+'\n')
print('\nFINAL COMPARISON:', PAIRED/'isles2024_paired_full_report.md')
print('JSON RESULTS:', PAIRED/'isles2024_paired_full_summary.json')
print('PAIRED PATIENT DATA:', PAIRED/'isles2024_paired_full_cases.csv')


## 6. Interpret results carefully

For **Dice** and **lesion-wise F1**, positive `perfusion_minus_ncct_mean` favors perfusion. For **absolute volume difference** and **absolute lesion-count difference**, negative values favor perfusion. Always quote both the paired 95% CI and **Holm-adjusted** p-values for the four metrics; an unadjusted p-value near 0.03 need not remain below 0.05 after correction. This is an **internal dataset ablation**, not an organizer-ranked hidden-test score.

If Colab stops mid-evaluation, rerun **Run all**. Existing validated case checkpoints remain in Google Drive and are reused when their inputs have not changed.
